# SmolVLM-256M: Benchmark

This notebook times one image with the bare model, eager decoding and CUDA-graph decoding. It also times the image preparation, with and without TensorRT for the vision encoder.

In [ ]:
import copy
import gc
import json
import re
import subprocess
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

ROOT = next(p for p in (Path(".."), Path("."), Path("/content/edge-ai-smol-lvm-250-optimization"))
            if (p / "frame.jpg").exists())
IMAGE = ROOT / "frame.jpg"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
PROMPT = "Describe any visible defects."
PREPROCESS = {"default": (None, "cpu"), "slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}


def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def mean_ms(fn, runs=5):
    fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs):
        fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / runs


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


def load(attn=None, fast=None):
    kw = {} if attn is None else {"attn_implementation": attn}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **({} if fast is None else {"use_fast": fast}))
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, **kw).to("cuda").eval()
    return model, proc


def eos_ids(tok):
    return {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}


def chat_text(proc, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    return proc.apply_chat_template(msgs, add_generation_prompt=True)


def make_inputs(proc, image, prompt, image_device="cpu", size=None, extra=None):
    img = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image
    dev = {} if image_device == "cpu" else {"device": image_device}
    ik = {**({} if size is None else {"size": {"longest_edge": size}}), **dev, **(extra or {})}
    text = chat_text(proc, prompt)
    try:
        out = proc(text=text, images=[img], return_tensors="pt", images_kwargs=ik)
    except (TypeError, ValueError):
        out = proc(text=text, images=[img], return_tensors="pt", **ik)
    out = out.to("cuda")
    out["pixel_values"] = out["pixel_values"].to(torch.float16)
    return out


def bare_generate(model, inputs, k, stop=True):
    extra = {} if stop else {"min_new_tokens": k}
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=k, do_sample=False, **extra)


class GraphDecoder:
    def __init__(self, model, max_new=64, max_len=4096, own_mask=False):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None
        self.steps = max_new
        self.own_mask = own_mask or model.config.text_config._attn_implementation == "eager"

    def mask(self):
        allowed = torch.arange(self.max_len, device=self.pos.device) <= self.pos[:, None]
        dt = self.model.dtype
        return ((~allowed).to(dt) * torch.finfo(dt).min)[:, None, None, :]

    def step(self):
        extra = {"attention_mask": self.mask()} if self.own_mask else {}
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True, **extra
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph=True, eos=None):
        self.steps = self.max_new
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                self.steps = i + 1
                break
        return self.out.clone()

    def generate(self, inputs, eos, use_graph=True):
        first = self.prefill(inputs)
        ids = torch.cat([first.view(1), self.decode(use_graph, eos)]).tolist()
        return until_eos(ids, eos)


def time_decode(dec, inputs, eos, runs=5, use_graph=True, stop=False):
    pre, dec_ms, ids = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        out, d = timed(lambda: dec.decode(use_graph, eos if stop else None))
        if i:
            pre.append(p)
            dec_ms.append(d)
        ids = until_eos(torch.cat([first.view(1), out]).tolist(), eos)
    return {"prefill_ms": float(np.mean(pre)), "decode_ms": float(np.mean(dec_ms)), "steps": dec.steps, "ids": ids}


def fit_len(n, max_new):
    return -(-(n + max_new + 1) // 64) * 64

In [ ]:
TRT = ROOT / "trt_siglip"


class SiglipTrt(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision = model.model.vision_model
        self.owner = [model]
        self.engines = {}

    def forward(self, pixel_values, patch_attention_mask=None, *args, **kwargs):
        if patch_attention_mask is not None and not bool(patch_attention_mask.all()):
            return self.vision(pixel_values, patch_attention_mask, *args, **kwargs)
        hidden = self.vision.embeddings(pixel_values, patch_attention_mask)
        n = hidden.shape[0]
        if n not in self.engines:
            self.engines[n] = engine_for(self.owner[0], self.vision, hidden)
        return BaseModelOutput(last_hidden_state=self.engines[n](hidden))


class SiglipEncoder(torch.nn.Module):
    def __init__(self, vision):
        super().__init__()
        self.encoder = vision.encoder
        self.norm = vision.post_layernorm

    def forward(self, x):
        return self.norm(self.encoder(inputs_embeds=x)[0])


def export_onnx(model, vision, x, path):
    set_attn(model, "eager")
    enc = copy.deepcopy(SiglipEncoder(vision)).half().eval()
    set_attn(model, "sdpa")
    with torch.no_grad():
        torch.onnx.export(enc, (x.half(),), str(path), dynamo=False,
                          input_names=["embeds"], output_names=["hidden"], opset_version=17)


def build_engine(onnx, engine):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    nf = trt.NetworkDefinitionCreationFlag
    has_fp16 = hasattr(trt.BuilderFlag, "FP16")
    flag = 1 << int(nf.EXPLICIT_BATCH) if hasattr(nf, "EXPLICIT_BATCH") else 0
    if not has_fp16:
        flag |= 1 << int(nf.STRONGLY_TYPED)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(Path(onnx).read_bytes()):
        raise RuntimeError("; ".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    if has_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    data = builder.build_serialized_network(network, config)
    if data is None:
        raise RuntimeError("TensorRT build failed")
    Path(engine).write_bytes(data)


class TrtEngine:
    def __init__(self, path):
        import tensorrt as trt

        engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(Path(path).read_bytes())
        self.ctx = engine.create_execution_context()
        names = [engine.get_tensor_name(i) for i in range(engine.num_io_tensors)]
        self.inp = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT)
        self.out = next(n for n in names if engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT)
        self.out_shape = tuple(engine.get_tensor_shape(self.out))
        dtypes = {trt.DataType.FLOAT: torch.float32, trt.DataType.HALF: torch.float16}
        self.in_dtype = dtypes[engine.get_tensor_dtype(self.inp)]
        self.out_dtype = dtypes[engine.get_tensor_dtype(self.out)]

    def __call__(self, x):
        x = x.to(self.in_dtype).contiguous()
        y = torch.empty(self.out_shape, dtype=self.out_dtype, device=x.device)
        self.ctx.set_tensor_address(self.inp, x.data_ptr())
        self.ctx.set_tensor_address(self.out, y.data_ptr())
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return y.half()


def engine_for(model, vision, hidden):
    TRT.mkdir(parents=True, exist_ok=True)
    n = hidden.shape[0]
    onnx, engine = TRT / f"siglip_{n}tiles.onnx", TRT / f"siglip_{n}tiles.engine"
    if not engine.exists():
        with torch.inference_mode(False):
            export_onnx(model, vision, hidden.clone(), onnx)
            build_engine(onnx, engine)
    return TrtEngine(engine)

In [ ]:
def bench(preprocess="fast-gpu", size=1024, trt=False, max_new=64, runs=5):
    fast, image_device = PREPROCESS[preprocess]
    model, proc = load(fast=fast)
    eos = eos_ids(proc.tokenizer)
    if trt:
        model.model.vision_model = SiglipTrt(model)
    text_ms = mean_ms(lambda: proc.tokenizer(chat_text(proc, PROMPT), return_tensors="pt"), runs)
    image_ms = mean_ms(lambda: Image.open(IMAGE).convert("RGB"), runs)
    input_ms = mean_ms(lambda: make_inputs(proc, IMAGE, PROMPT, image_device, size), runs)
    inputs = make_inputs(proc, IMAGE, PROMPT, image_device, size)
    n = inputs["input_ids"].shape[1]
    dec = GraphDecoder(model, max_new, fit_len(n, max_new))
    dec.capture()

    bare_pre = mean_ms(lambda: bare_generate(model, inputs, 1, stop=False), runs)
    bare_total = mean_ms(lambda: bare_generate(model, inputs, max_new, stop=False), runs)
    bare_ids = bare_generate(model, inputs, max_new, stop=False)[0, n:].tolist()
    graph = time_decode(dec, inputs, eos, runs, use_graph=True)
    eager = time_decode(dec, inputs, eos, runs, use_graph=False)
    stop = time_decode(dec, inputs, eos, runs, use_graph=True, stop=True)

    return {
        "preprocess": preprocess, "tiles": inputs["pixel_values"].shape[1], "tokens": n, "trt": trt,
        "image ms": round(image_ms, 1), "tokenize ms": round(text_ms, 1), "input total ms": round(input_ms, 1),
        "bare prefill ms": round(bare_pre, 1), "bare ms/token": round((bare_total - bare_pre) / max_new, 2),
        "eager prefill ms": round(eager["prefill_ms"], 1), "eager ms/token": round(eager["decode_ms"] / max_new, 2),
        "graph prefill ms": round(graph["prefill_ms"], 1), "graph ms/token": round(graph["decode_ms"] / max_new, 2),
        "stop steps": stop["steps"], "stop decode ms": round(stop["decode_ms"], 1),
        "request 64 steps ms": round(input_ms + bare_total, 1),
        "request graph 64 ms": round(input_ms + graph["prefill_ms"] + graph["decode_ms"], 1),
        "request stop ms": round(input_ms + stop["prefill_ms"] + stop["decode_ms"], 1),
        "graph = eager": graph["ids"] == eager["ids"],
        "bare matches graph": bare_ids[: len(graph["ids"])] == graph["ids"],
    }


rows = [bench(trt=False), bench(trt=True)]
pd.DataFrame(rows).set_index("trt").T

## Log of previous runs

The cells below are the earlier notebook with their outputs. They call the scripts that were replaced by the code above, so they are not run again.

In [18]:
REPO = "./edge-ai-smol-lvm-250-optimization"
MAX_LEN = 1024

In [19]:
import subprocess

for option in ["slow", "fast-cpu", "fast-gpu"]:
    r = subprocess.run(
        ["python", f"{REPO}/bench.py", "--image", f"{REPO}/frame.jpg",
         "--max-len", str(MAX_LEN), "--preprocess", option],
        capture_output=True, text=True,
    )
    print(f"=== preprocess: {option}")
    print(r.stdout if r.returncode == 0 else r.stderr.strip().splitlines()[-1])

=== preprocess: slow
preprocess: slow (Idefics3ImageProcessorPil, image on cpu)
memory after load: RAM peak 1822 MB | VRAM allocated 497 MB, peak 497 MB, reserved 520 MB
prompt tokens: 875 | max_len: 1024
preprocessing (JPEG decode and tokenize on CPU, tile + normalize on CPU):
  JPEG decode                                0.9 ms
  tile + normalize                         295.8 ms
  tokenize                                   0.1 ms
  make_inputs total (incl. copy to GPU)    314.3 ms
 bare: prefill  280.6 ms | decode 39.95 ms/token |   25.0 tok/s
eager: prefill  272.0 ms | decode 32.16 ms/token |   31.1 tok/s
graph: prefill  272.0 ms | decode  8.14 ms/token |  122.8 tok/s
identical outputs (eager vs graph): True
bare matches graph up to EOS: True
memory after benchmark: RAM peak 2228 MB | VRAM allocated 591 MB, peak 873 MB, reserved 1110 MB

=== preprocess: fast-cpu
preprocess: fast-cpu (Idefics3ImageProcessor, image on cpu)
memory after load: RAM peak 1823 MB | VRAM allocated 497 MB, pe

## Result

The test ran on a Colab GPU with one image. The prompt has 875 tokens, max_len is 1024, and the request makes 64 new tokens. All answer checks are True.

**Graph decoding, for each image processor option**

| Option | Image processor | Tile + normalize (ms) | Input total (ms) | Prefill (ms) | Decode (ms/token) | Request total (ms) | Preprocessing share |
|---|---|---|---|---|---|---|---|
| slow | Idefics3ImageProcessorPil | 295.8 | 314.3 | 272.0 | 8.14 | 1107 | 28% |
| fast-cpu | Idefics3ImageProcessor | 96.0 | 116.8 | 283.0 | 8.16 | 922 | 13% |
| fast-gpu | Idefics3ImageProcessor | 12.8 | 14.1 | 284.4 | 8.19 | 823 | 2% |

**Request total (ms) for each decoding mode**

| Mode | slow | fast-cpu | fast-gpu |
|---|---|---|---|
| bare | 3152 | 2877 | 2810 |
| eager | 2645 | 2405 | 2311 |
| graph | 1107 | 922 | 823 |

The request total is the input total, the prefill, and 64 decode steps.

- `fast-gpu` cuts the input preparation from 314 ms to 14 ms. With graph decoding, the request falls from 1107 ms to 823 ms (26% less).
- Graph decoding and `fast-gpu` together take the request from 3152 ms (bare model, `slow`) to 823 ms. This is 3.8 times faster.
- The fast processor on the CPU gives about half of the gain (1107 ms to 922 ms). The GPU gives the rest.
- Prefill is about 11 ms slower with both fast options (283 ms and 284 ms, against 272 ms). We do not know the cause. This is small against the 300 ms gain.
- Decode times do not depend on the image processor.
- Peak VRAM is 873 MB in all three runs. The prefill sets the peak, so GPU preprocessing does not raise it. The reserved memory is 22 MB higher with `fast-gpu` (1132 MB and 1110 MB). The earlier +146 MB result read the peak before the prefill ran.
- With `fast-gpu`, the request is now 64% decode (524 ms), 35% prefill (284 ms) and 2% preprocessing (14 ms). Decode is the next target.

In [23]:
# best setup: fast-gpu preprocessing, graph decoding, with and without stop at EOS
import subprocess

r = subprocess.run(
    ["python", f"{REPO}/bench.py", "--image", f"{REPO}/frame.jpg",
     "--max-len", str(MAX_LEN), "--preprocess", "fast-gpu"],
    capture_output=True, text=True,
)
if r.returncode:
    print(r.stderr.strip().splitlines()[-1])
else:
    keys = ("make_inputs total", "graph:", "stop at EOS", "request")
    print("\n".join(l for l in r.stdout.splitlines() if any(k in l for k in keys)))

  make_inputs total (incl. copy to GPU)     15.0 ms
graph: prefill  270.4 ms | decode  8.14 ms/token |  122.8 tok/s
graph + stop at EOS: prefill  272.5 ms | decode   81.4 ms for 9 steps
request (input + prefill + decode), graph: 806 ms with 64 steps | 369 ms with stop at EOS (9 steps)


## Result: stop at EOS

The test ran on a Colab GPU with one image, `fast-gpu` preprocessing, graph decoding and max_len 1024. The answer has 9 tokens.

| Setup | Input (ms) | Prefill (ms) | Decode (ms) | Request total (ms) |
|---|---|---|---|---|
| 64 steps | 16.1 | 269.3 | 520 (64 x 8.13) | 806 |
| Stop at EOS (9 steps) | 16.1 | 271.8 | 80.9 | 369 |

- Stop at EOS cuts the request from 806 ms to 369 ms. This is 54% less, or 2.2 times faster.
- The result matches the EOS experiment: 80.9 ms for 9 steps against 79.2 ms.
- Decode takes about 9 ms for each step with only 9 steps. A fixed time of about 7 ms for each decode call is spread over few steps. A longer answer costs about 8.1 ms for each extra token.
- Prefill is now 74% of the request. Decode is 22% and the input is 4%.
- The first full request took 1107 ms (slow processor, 64 steps). It now takes 369 ms, which is 3.0 times faster.
- Prefill is the next target: fewer tiles, cached vision embeddings, and a TensorRT engine for SigLIP. Frame pipelining would hide only 16 ms of CPU work, so it now gains little.

In [ ]:
# best setup: size 1024 (5 tiles), fast-gpu preprocessing, graph decoding, stop at EOS, TensorRT engine for SigLIP
# Colab only: install the packages. The engine is built on the first run (about 30 s) and reused after that.
!pip install -q tensorrt onnx

import subprocess

r = subprocess.run(
    ["python", f"{REPO}/bench.py", "--image", f"{REPO}/frame.jpg", "--max-len", "448",
     "--preprocess", "fast-gpu", "--size", "1024", "--trt"],
    capture_output=True, text=True,
)
if r.returncode:
    print(r.stderr.strip().splitlines()[-1])
else:
    keys = ("prompt tokens", "make_inputs total", "graph:", "stop at EOS", "request", "TensorRT")
    print("\n".join(l for l in r.stdout.splitlines() if any(k in l for k in keys)))

## Result: fewer tiles and TensorRT

The test ran on a Colab GPU with one image, `--size 1024` (5 tiles, 346 prompt tokens), `fast-gpu` preprocessing and graph decoding with max_len 448. The answer has 9 tokens.

| Setup | Input (ms) | Prefill (ms) | Decode (ms) | Request total (ms) |
|---|---|---|---|---|
| 64 steps | 5.5 | 102.1 | 425 (64 x 6.64) | 533 |
| Stop at EOS | 5.5 | 101.9 | 65.8 | 173 |
| Stop at EOS + TensorRT | 5.5 | 83.4 | 70.3 | 159 |

- With 5 tiles and stop at EOS, the request takes 173 ms. The setup with 13 tiles took 369 ms. This is 2.1 times faster. The tile test showed the same answer for this image.
- TensorRT cuts the prefill from 101.9 ms to 83.4 ms (18.5 ms, 18% less). The answer is the same.
- The decode is 4.5 ms slower with TensorRT (70.3 ms and 65.8 ms). TensorRT does not change the decode, so this is probably noise.
- The net gain of TensorRT is 14 ms (8%): 173 ms to 159 ms. With 13 tiles, the gain was 32 ms. SigLIP has fewer tiles now, so there is less to save.
- The request is now 52% prefill, 44% decode and 3% input. Decode is a large part again.
- The request is about 6 times faster than the bare model (about 954 ms, estimated from separate measurements).
- Next steps: shorter answers, because each token costs about 6.6 ms, and a test on frames with known defects, because fewer tiles lower the resolution.